In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit193_aug_1.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit488_aug_2.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit224_aug_2.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit396_aug_0.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit492_aug_1.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit293_aug_2.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit344_aug_0.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit2_aug_2.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/val/labels/circuit180_aug_1.txt
/kaggle/input/yolo-text-detection-data2/YOLO_text_dete

In [1]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 66.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 54.5 MB/s eta 0:00:0000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 952.0 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 18.8 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: n

In [2]:
import time
import json
import pandas as pd
from ultralytics import YOLO
from pathlib import Path

# =============================
# CONFIG
# =============================
DATA_PATH = "/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml"
RESULTS_CSV = "yolov8_ablation_results.csv"

# Global Best Parameters (update as you go)
BEST_PARAMS = {
    "epochs": 32,          # default, update after finding best
    "batch": 32,
    "lr0": 0.01,
    "weight_decay": 0.0005,
    "optimizer": "SGD",
    "img_size": 640,
    "dropout": 0.0,
    "freeze": 0,
    "activation": "SiLU"
}

# Initialize results DataFrame
if Path(RESULTS_CSV).exists():
    results_df = pd.read_csv(RESULTS_CSV)
else:
    results_df = pd.DataFrame(columns=[
        "Case", "Exp No", "Experiment Name", "Epochs", "Batch", "lr0", "Weight Decay",
        "Optimizer", "Image Size", "Dropout", "Freeze Layers", "Activation",
        "Training Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"
    ])


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
def run_experiment(case, exp_no, exp_name, epochs=32, batch=32, lr0=0.01,
                   weight_decay=0.0005, optimizer="SGD", img_size=640,
                   dropout=0.0, freeze=0, activation="SiLU"):
    global results_df

    model = YOLO("yolov8n.pt")  # Using YOLOv8 nano

    # Track training time
    start_time = time.time()
    model.train(
        data=DATA_PATH,
        epochs=epochs,
        batch=batch,
        lr0=lr0,
        optimizer=optimizer,
        imgsz=img_size,
        dropout=dropout,
        freeze=freeze,
        verbose=False
    )
    train_time = time.time() - start_time

    # ✅ FIX: Load YOLO metrics from results.csv
    results_file = Path(model.trainer.save_dir) / "results.csv"
    results_df_yolo = pd.read_csv(results_file)
    last_row = results_df_yolo.iloc[-1]

    precision = last_row["metrics/precision(B)"]
    recall = last_row["metrics/recall(B)"]
    map50 = last_row["metrics/mAP50(B)"]
    map5095 = last_row["metrics/mAP50-95(B)"]

    new_row = {
        "Case": case,
        "Exp No": exp_no,
        "Experiment Name": exp_name,
        "Epochs": epochs,
        "Batch": batch,
        "lr0": lr0,
        "Weight Decay": weight_decay,
        "Optimizer": optimizer,
        "Image Size": img_size,
        "Dropout": dropout,
        "Freeze Layers": freeze,
        "Activation": activation,
        "Training Time (s)": round(train_time, 2),
        "Precision": precision,
        "Recall": recall,
        "mAP@0.5": map50,
        "mAP@0.5:0.95": map5095,
    }
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

    results_df.to_csv(RESULTS_CSV, index=False)
    display(results_df.tail(10))
# =============================
# Wrapper to use BEST_PARAMS
# =============================
def run_with_best(case, exp_no, exp_name, **kwargs):
    params = BEST_PARAMS.copy()
    params.update(kwargs)
    run_experiment(case, exp_no, exp_name, **params)

In [7]:
# Case 1: Epoch Variation
def case1_epoch_variation():
    run_with_best("Case 1: Epoch Variation", 1, "epochs_16", epochs=16)
    run_with_best("Case 1: Epoch Variation", 2, "epochs_32", epochs=32)
    run_with_best("Case 1: Epoch Variation", 3, "epochs_64", epochs=64)

# Case 2: Weight Decay
def case2_weight_decay():
    run_with_best("Case 2: Weight Decay", 1, "wd_0", weight_decay=0)
    run_with_best("Case 2: Weight Decay", 2, "wd_0.0001", weight_decay=0.0001)
    run_with_best("Case 2: Weight Decay", 3, "wd_0.0005", weight_decay=0.0005)
    run_with_best("Case 2: Weight Decay", 4, "wd_0.001", weight_decay=0.001)
    run_with_best("Case 2: Weight Decay", 5, "wd_0.005", weight_decay=0.005)

# Case 3: Batch Size
def case3_batch_size():
    run_with_best("Case 3: Batch Size", 1, "batch_16", batch=16)
    run_with_best("Case 3: Batch Size", 2, "batch_32", batch=32)
    run_with_best("Case 3: Batch Size", 3, "batch_64", batch=64)  # may OOM

# Case 4: Dropout
def case4_dropout():
    for i, d in enumerate([0, 0.1, 0.2, 0.3, 0.5], start=1):
        run_with_best("Case 4: Dropout", i, f"dropout_{d}", dropout=d)

# Case 6: Optimizer
def case6_optimizer():
    opts = ["SGD", "Adam", "AdamW", "RMSProp", "Adamax"]
    for i, opt in enumerate(opts, start=1):
        run_with_best("Case 6: Optimizer", i, f"optimizer_{opt}", optimizer=opt)
# Case 7: Learning Rate
def case7_learning_rate():
    for i, lr in enumerate([0.001, 0.005, 0.01, 0.05], start=1):
        run_with_best("Case 7: Learning Rate", i, f"lr_{lr}", lr0=lr)

# Case 8: Freeze Layers
def case8_freeze_layers():
    for i, freeze in enumerate([0, 10, 15, 22], start=1):
        run_with_best("Case 8: Freeze Layers", i, f"freeze_{freeze}", freeze=freeze)
# Case 5: Image Size
# Case 5: Image Size
def case5_image_size():
    for i, img_size in enumerate([320, 512, 640], start=1):
        run_with_best("Case 5: Image Size", i, f"imgsz_{img_size}", img_size=img_size)




In [20]:
# =============================
# Case 9: Activation Functions
# =============================

# !pip install ultralytics

import time, yaml, copy
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from ultralytics.nn.tasks import yaml_model_load
from tabulate import tabulate

# -------------------------------
# CONFIG
# -------------------------------
DATA_PATH = "/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml"
RESULTS_CSV = "case9_results.csv"
BASE_MODEL = "yolov8n.yaml"   # change to yolov8m.yaml if you want bigger model

# -------------------------------
# Initialize results file
# -------------------------------
if Path(RESULTS_CSV).exists():
    results_df = pd.read_csv(RESULTS_CSV)
else:
    results_df = pd.DataFrame(columns=[
        "Activation", "Epochs", "Batch", "Optimizer", "lr0", "Weight Decay",
        "Image Size", "Freeze", "Training Time (s)",
        "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"
    ])

# -------------------------------
# Baseline training params
# -------------------------------
BASE_PARAMS = {
    "epochs": 32,      # small for demo, increase later
    "batch": 32,
    "lr0": 0.05,
    "weight_decay": 0.0005,
    "optimizer": "SGD",
    "imgsz": 640,
    "freeze": 0
}

# -------------------------------
# Helper: Create new YAML with activation
# -------------------------------
def create_activation_yaml(base_yaml, act_expr, suffix):
    cfg = yaml_model_load(base_yaml)   # loads YOLO config from ultralytics
    cfg["activation"] = act_expr

    out_path = f"/kaggle/working/{Path(base_yaml).stem}_{suffix}.yaml"
    with open(out_path, "w") as f:
        yaml.dump(cfg, f)

    return out_path

# -------------------------------
# Case 9: Activation Functions
# -------------------------------
activations = {
    "ReLU": ("nn.ReLU()", "relu"),
    "LeakyReLU": ("nn.LeakyReLU(0.1)", "leakyrelu"),
    "Mish": ("nn.Mish()", "mish"),
    "SiLU": ("nn.SiLU()", "silu")
}

all_results = []

for act_name, (act_expr, suffix) in activations.items():
    print(f"\n🚀 Training with {act_name} activation...")

    # Create activation YAML
    yaml_path = create_activation_yaml(BASE_MODEL, act_expr, suffix)

    # Init model from YAML
    model = YOLO(yaml_path)

    # Train
    start = time.time()
    model.train(
        data=DATA_PATH,
        epochs=BASE_PARAMS["epochs"],
        batch=BASE_PARAMS["batch"],
        lr0=BASE_PARAMS["lr0"],
        optimizer=BASE_PARAMS["optimizer"],
        imgsz=BASE_PARAMS["imgsz"],
        weight_decay=BASE_PARAMS["weight_decay"],
        freeze=BASE_PARAMS["freeze"],
        verbose=False
    )
    train_time = time.time() - start

    # Load YOLO's results.csv (last row)
    results_file = Path(model.trainer.save_dir) / "results.csv"
    df_metrics = pd.read_csv(results_file)
    last = df_metrics.iloc[-1]

    result = {
        "Activation": act_name,
        "Epochs": BASE_PARAMS["epochs"],
        "Batch": BASE_PARAMS["batch"],
        "Optimizer": BASE_PARAMS["optimizer"],
        "lr0": BASE_PARAMS["lr0"],
        "Weight Decay": BASE_PARAMS["weight_decay"],
        "Image Size": BASE_PARAMS["imgsz"],
        "Freeze": BASE_PARAMS["freeze"],
        "Training Time (s)": round(train_time, 2),
        "Precision": last["metrics/precision(B)"],
        "Recall": last["metrics/recall(B)"],
        "mAP@0.5": last["metrics/mAP50(B)"],
        "mAP@0.5:0.95": last["metrics/mAP50-95(B)"]
    }

    results_df = pd.concat([results_df, pd.DataFrame([result])], ignore_index=True)
    results_df.to_csv(RESULTS_CSV, index=False)

    all_results.append(result)

# -------------------------------
# Show final results
# -------------------------------
print("\n📊 Case 9 Results:\n")
print(tabulate(pd.DataFrame(all_results), headers="keys", tablefmt="grid"))



🚀 Training with ReLU activation...
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/yolov8n_relu.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, 

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798       0.99      0.991      0.993      0.698
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/train4

🚀 Training with LeakyReLU activation...
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, k

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.989      0.992      0.993        0.7
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to /kaggle/working/runs/detect/train5

🚀 Training with Mish activation...
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.995      0.989      0.995        0.7
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to /kaggle/working/runs/detect/train6

🚀 Training with SiLU activation...
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.994      0.992      0.994      0.706
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 3.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/train7

📊 Case 9 Results:

+----+--------------+----------+---------+-------------+-------+----------------+--------------+----------+---------------------+-------------+----------+-----------+----------------+
|    | Activation   |   Epochs |   Batch | Optimizer   |   lr0 |   Weight Decay |   Image Size |   Freeze |   Training Time (s) |   Precision |   Recall |   mAP@0.5 |   mAP@0.5:0.95 |
+====+==============+==========+=========+=============+=======+================+==============+==========+=====================+=============+==========+===========+================+
|  0 | ReLU         |       32 |      32 | SGD         |  0.05 |         0.0005 |          640 |        0 |              452.9  |     0.98989 |  0.99055 |   0.99332 |        0.69765 |
+----+--------------+

In [16]:
import yaml, copy

def create_activation_yaml(base_yaml="yolov8n.yaml", act_expr="nn.SiLU()", suffix="silu"):
    """
    Create a modified YOLOv8 YAML with a custom activation function.
    Returns the path to the new YAML.
    """
    with open(base_yaml, "r") as f:
        cfg = yaml.safe_load(f)

    # Set activation
    cfg["activation"] = act_expr

    # Build safe filename
    out_path = f"/kaggle/working/{Path(base_yaml).stem}_{suffix}.yaml"
    with open(out_path, "w") as f:
        yaml.dump(cfg, f)

    return out_path


In [17]:
# Case 9: Activation Functions
def case9_activation_functions():
    activations = {
        "ReLU": ("nn.ReLU()", "relu"),
        "LeakyReLU": ("nn.LeakyReLU(0.1)", "leakyrelu"),
        "Mish": ("nn.Mish()", "mish"),
        "SiLU": ("nn.SiLU()", "silu")  # baseline
    }

    for i, (act_name, (act_expr, suffix)) in enumerate(activations.items(), start=1):
        # Create activation-specific yaml
        yaml_path = create_activation_yaml("yolov8n.yaml", act_expr, suffix)

        # Temporarily override model init
        model = YOLO(yaml_path)

        # Run experiment (reuse logging)
        run_with_best("Case 9: Activation Functions", i, f"activation_{act_name}", activation=act_name)


In [18]:
case9_activation_functions()


FileNotFoundError: [Errno 2] No such file or directory: 'yolov8n.yaml'

In [8]:
case5_image_size()

Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=1

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.997      0.993      0.994      0.719
Speed: 0.0ms preprocess, 0.4ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/train


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 5: Image Size,1,imgsz_320,32,32,0.01,0.0005,SGD,320,0.0,0,SiLU,226.53,0.99776,0.9931,0.99448,0.71656


Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

...

,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 5: Image Size,1,imgsz_320,32,32,0.01,0.0005,SGD,320,0.0,0,SiLU,226.53,0.99776,0.99310,0.99448,0.71656
1,Case 5: Image Size,2,imgsz_512,32,32,0.01,0.0005,SGD,512,0.0,0,SiLU,348.45,0.99701,0.99388,0.99464,0.72444


Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.994      0.995      0.732
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/train3


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 5: Image Size,1,imgsz_320,32,32,0.01,0.0005,SGD,320,0.0,0,SiLU,226.53,0.99776,0.99310,0.99448,0.71656
1,Case 5: Image Size,2,imgsz_512,32,32,0.01,0.0005,SGD,512,0.0,0,SiLU,348.45,0.99701,0.99388,0.99464,0.72444
2,Case 5: Image Size,3,imgsz_640,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,456.79,0.99622,0.99444,0.99470,0.73197


In [9]:
import copy, yaml

def create_activation_yaml(base_yaml="yolov8n.yaml", act_expr="nn.SiLU()", suffix="silu"):
    """
    Create a modified YOLOv8 YAML with a custom activation function.
    Returns path to the new YAML.
    """
    with open(base_yaml, "r") as f:
        cfg = yaml.safe_load(f)

    # Update activation in model definition
    cfg["activation"] = act_expr

    out_path = f"/kaggle/working/{Path(base_yaml).stem}_{suffix}.yaml"
    with open(out_path, "w") as f:
        yaml.dump(cfg, f)

    return out_path


In [12]:
!cp /usr/local/lib/python3.10/dist-packages/ultralytics/cfg/models/v8/yolov8n.yaml /kaggle/working/


cp: cannot stat '/usr/local/lib/python3.10/dist-packages/ultralytics/cfg/models/v8/yolov8n.yaml': No such file or directory


In [13]:
# Case 9: Activation Functions
def case9_activation_functions():
    activations = {
        "ReLU": ("nn.ReLU()", "relu"),
        "LeakyReLU": ("nn.LeakyReLU(0.1)", "leakyrelu"),
        "Mish": ("nn.Mish()", "mish"),
        "SiLU": ("nn.SiLU()", "silu")  # baseline
    }

    for i, (act_name, (act_expr, suffix)) in enumerate(activations.items(), start=1):
        yaml_path = create_activation_yaml("yolov8n.yaml", act_expr, suffix)

        # Train with the modified YAML
        model = YOLO(yaml_path)

        run_with_best("Case 9: Activation Functions", i, f"activation_{act_name}", activation=act_name)


In [14]:
case9_activation_functions()

FileNotFoundError: [Errno 2] No such file or directory: 'yolov8n.yaml'

In [23]:
case1_epoch_variation()

Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=16, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.997      0.995      0.995      0.707
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to runs/detect/train2


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,238.37,0.99721,0.99478,0.9946,0.70744


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

...

,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,238.37,0.99721,0.99478,0.9946,0.70744
1,Case 1: Epoch Variation,2,epochs_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,458.60,0.99622,0.99444,0.9947,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=64, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.994      0.997      0.995      0.758
Speed: 0.1ms preprocess, 2.0ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to runs/detect/train4


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,238.37,0.99721,0.99478,0.99460,0.70744
1,Case 1: Epoch Variation,2,epochs_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,458.60,0.99622,0.99444,0.99470,0.73197
2,Case 1: Epoch Variation,3,epochs_64,64,32,0.01,0.0005,SGD,640,0.0,0,SiLU,902.30,0.99784,0.99333,0.99432,0.75650


In [24]:
BEST_PARAMS["epochs"] = 32
case2_weight_decay()

Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.994      0.995      0.732
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to runs/detect/train5


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,238.37,0.99721,0.99478,0.99460,0.70744
1,Case 1: Epoch Variation,2,epochs_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,458.60,0.99622,0.99444,0.99470,0.73197
2,Case 1: Epoch Variation,3,epochs_64,64,32,0.01,0.0005,SGD,640,0.0,0,SiLU,902.30,0.99784,0.99333,0.99432,0.75650
3,Case 2: Weight Decay,1,wd_0,32,32,0.01,0.0000,SGD,640,0.0,0,SiLU,458.41,0.99622,0.99444,0.99470,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

...

,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,238.37,0.99721,0.99478,0.99460,0.70744
1,Case 1: Epoch Variation,2,epochs_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,458.60,0.99622,0.99444,0.99470,0.73197
2,Case 1: Epoch Variation,3,epochs_64,64,32,0.01,0.0005,SGD,640,0.0,0,SiLU,902.30,0.99784,0.99333,0.99432,0.75650
3,Case 2: Weight Decay,1,wd_0,32,32,0.01,0.0000,SGD,640,0.0,0,SiLU,458.41,0.99622,0.99444,0.99470,0.73197
4,Case 2: Weight Decay,2,wd_0.0001,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,458.25,0.99622,0.99444,0.99470,0.73197
5,Case 2: Weight Decay,3,wd_0.0005,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,460.87,0.99622,0.99444,0.99470,0.73197
6,Case 2: Weight Decay,4,wd_0.001,32,32,0.01,0.0010,SGD,640,0.0,0,SiLU,463.15,0.99622,0.99444,0.99470,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train9, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.994      0.995      0.732
Speed: 0.1ms preprocess, 1.8ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to runs/detect/train9


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,238.37,0.99721,0.99478,0.99460,0.70744
1,Case 1: Epoch Variation,2,epochs_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,458.60,0.99622,0.99444,0.99470,0.73197
2,Case 1: Epoch Variation,3,epochs_64,64,32,0.01,0.0005,SGD,640,0.0,0,SiLU,902.30,0.99784,0.99333,0.99432,0.75650
3,Case 2: Weight Decay,1,wd_0,32,32,0.01,0.0000,SGD,640,0.0,0,SiLU,458.41,0.99622,0.99444,0.99470,0.73197
4,Case 2: Weight Decay,2,wd_0.0001,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,458.25,0.99622,0.99444,0.99470,0.73197
5,Case 2: Weight Decay,3,wd_0.0005,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,460.87,0.99622,0.99444,0.99470,0.73197
6,Case 2: Weight Decay,4,wd_0.001,32,32,0.01,0.0010,SGD,640,0.0,0,SiLU,463.15,0.99622,0.99444,0.99470,0.73197
7,Case 2: Weight Decay,5,wd_0.005,32,32,0.01,0.0050,SGD,640,0.0,0,SiLU,463.39,0.99622,0.99444,0.99470,0.73197


In [25]:
BEST_PARAMS["epochs"] = 32
BEST_PARAMS["weight_decay"] = 0.0001
case3_batch_size()


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train10, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.994      0.996      0.994      0.732
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to runs/detect/train10


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,238.37,0.99721,0.99478,0.99460,0.70744
1,Case 1: Epoch Variation,2,epochs_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,458.60,0.99622,0.99444,0.99470,0.73197
2,Case 1: Epoch Variation,3,epochs_64,64,32,0.01,0.0005,SGD,640,0.0,0,SiLU,902.30,0.99784,0.99333,0.99432,0.75650
3,Case 2: Weight Decay,1,wd_0,32,32,0.01,0.0000,SGD,640,0.0,0,SiLU,458.41,0.99622,0.99444,0.99470,0.73197
4,Case 2: Weight Decay,2,wd_0.0001,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,458.25,0.99622,0.99444,0.99470,0.73197
5,Case 2: Weight Decay,3,wd_0.0005,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,460.87,0.99622,0.99444,0.99470,0.73197
6,Case 2: Weight Decay,4,wd_0.001,32,32,0.01,0.0010,SGD,640,0.0,0,SiLU,463.15,0.99622,0.99444,0.99470,0.73197
7,Case 2: Weight Decay,5,wd_0.005,32,32,0.01,0.0050,SGD,640,0.0,0,SiLU,463.39,0.99622,0.99444,0.99470,0.73197
8,Case 3: Batch Size,1,batch_16,32,16,0.01,0.0001,SGD,640,0.0,0,SiLU,515.74,0.99389,0.99555,0.99444,0.73239


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train11, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

...

,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 1: Epoch Variation,1,epochs_16,16,32,0.01,0.0005,SGD,640,0.0,0,SiLU,238.37,0.99721,0.99478,0.99460,0.70744
1,Case 1: Epoch Variation,2,epochs_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,458.60,0.99622,0.99444,0.99470,0.73197
2,Case 1: Epoch Variation,3,epochs_64,64,32,0.01,0.0005,SGD,640,0.0,0,SiLU,902.30,0.99784,0.99333,0.99432,0.75650
3,Case 2: Weight Decay,1,wd_0,32,32,0.01,0.0000,SGD,640,0.0,0,SiLU,458.41,0.99622,0.99444,0.99470,0.73197
4,Case 2: Weight Decay,2,wd_0.0001,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,458.25,0.99622,0.99444,0.99470,0.73197
5,Case 2: Weight Decay,3,wd_0.0005,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,460.87,0.99622,0.99444,0.99470,0.73197
6,Case 2: Weight Decay,4,wd_0.001,32,32,0.01,0.0010,SGD,640,0.0,0,SiLU,463.15,0.99622,0.99444,0.99470,0.73197
7,Case 2: Weight Decay,5,wd_0.005,32,32,0.01,0.0050,SGD,640,0.0,0,SiLU,463.39,0.99622,0.99444,0.99470,0.73197
8,Case 3: Batch Size,1,batch_16,32,16,0.01,0.0001,SGD,640,0.0,0,SiLU,515.74,0.99389,0.99555,0.99444,0.73239
9,Case 3: Batch Size,2,batch_32,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,465.39,0.99622,0.99444,0.99470,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train12, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.995      0.996      0.995      0.731
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to runs/detect/train12


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
1,Case 1: Epoch Variation,2,epochs_32,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,458.60,0.99622,0.99444,0.99470,0.73197
2,Case 1: Epoch Variation,3,epochs_64,64,32,0.01,0.0005,SGD,640,0.0,0,SiLU,902.30,0.99784,0.99333,0.99432,0.75650
3,Case 2: Weight Decay,1,wd_0,32,32,0.01,0.0000,SGD,640,0.0,0,SiLU,458.41,0.99622,0.99444,0.99470,0.73197
4,Case 2: Weight Decay,2,wd_0.0001,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,458.25,0.99622,0.99444,0.99470,0.73197
5,Case 2: Weight Decay,3,wd_0.0005,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,460.87,0.99622,0.99444,0.99470,0.73197
6,Case 2: Weight Decay,4,wd_0.001,32,32,0.01,0.0010,SGD,640,0.0,0,SiLU,463.15,0.99622,0.99444,0.99470,0.73197
7,Case 2: Weight Decay,5,wd_0.005,32,32,0.01,0.0050,SGD,640,0.0,0,SiLU,463.39,0.99622,0.99444,0.99470,0.73197
8,Case 3: Batch Size,1,batch_16,32,16,0.01,0.0001,SGD,640,0.0,0,SiLU,515.74,0.99389,0.99555,0.99444,0.73239
9,Case 3: Batch Size,2,batch_32,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,465.39,0.99622,0.99444,0.99470,0.73197
10,Case 3: Batch Size,3,batch_64,32,64,0.01,0.0001,SGD,640,0.0,0,SiLU,449.77,0.99500,0.99605,0.99455,0.73081


In [ ]:
BEST_PARAMS["epochs"] = 32
BEST_PARAMS["weight_decay"] = 0.0001
BEST_PARAMS["batch"] = 32
case4_dropout()


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train13, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=1

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.994      0.995      0.732
Speed: 0.1ms preprocess, 2.0ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to runs/detect/train13


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
2,Case 1: Epoch Variation,3,epochs_64,64,32,0.01,0.0005,SGD,640,0.0,0,SiLU,902.30,0.99784,0.99333,0.99432,0.75650
3,Case 2: Weight Decay,1,wd_0,32,32,0.01,0.0000,SGD,640,0.0,0,SiLU,458.41,0.99622,0.99444,0.99470,0.73197
4,Case 2: Weight Decay,2,wd_0.0001,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,458.25,0.99622,0.99444,0.99470,0.73197
5,Case 2: Weight Decay,3,wd_0.0005,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,460.87,0.99622,0.99444,0.99470,0.73197
6,Case 2: Weight Decay,4,wd_0.001,32,32,0.01,0.0010,SGD,640,0.0,0,SiLU,463.15,0.99622,0.99444,0.99470,0.73197
7,Case 2: Weight Decay,5,wd_0.005,32,32,0.01,0.0050,SGD,640,0.0,0,SiLU,463.39,0.99622,0.99444,0.99470,0.73197
8,Case 3: Batch Size,1,batch_16,32,16,0.01,0.0001,SGD,640,0.0,0,SiLU,515.74,0.99389,0.99555,0.99444,0.73239
9,Case 3: Batch Size,2,batch_32,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,465.39,0.99622,0.99444,0.99470,0.73197
10,Case 3: Batch Size,3,batch_64,32,64,0.01,0.0001,SGD,640,0.0,0,SiLU,449.77,0.99500,0.99605,0.99455,0.73081
11,Case 4: Dropout,1,dropout_0,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,468.16,0.99622,0.99444,0.99470,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train14, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.994      0.995      0.732
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to runs/detect/train14


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
3,Case 2: Weight Decay,1,wd_0,32,32,0.01,0.0000,SGD,640,0.0,0,SiLU,458.41,0.99622,0.99444,0.99470,0.73197
4,Case 2: Weight Decay,2,wd_0.0001,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,458.25,0.99622,0.99444,0.99470,0.73197
5,Case 2: Weight Decay,3,wd_0.0005,32,32,0.01,0.0005,SGD,640,0.0,0,SiLU,460.87,0.99622,0.99444,0.99470,0.73197
6,Case 2: Weight Decay,4,wd_0.001,32,32,0.01,0.0010,SGD,640,0.0,0,SiLU,463.15,0.99622,0.99444,0.99470,0.73197
7,Case 2: Weight Decay,5,wd_0.005,32,32,0.01,0.0050,SGD,640,0.0,0,SiLU,463.39,0.99622,0.99444,0.99470,0.73197
8,Case 3: Batch Size,1,batch_16,32,16,0.01,0.0001,SGD,640,0.0,0,SiLU,515.74,0.99389,0.99555,0.99444,0.73239
9,Case 3: Batch Size,2,batch_32,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,465.39,0.99622,0.99444,0.99470,0.73197
10,Case 3: Batch Size,3,batch_64,32,64,0.01,0.0001,SGD,640,0.0,0,SiLU,449.77,0.99500,0.99605,0.99455,0.73081
11,Case 4: Dropout,1,dropout_0,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,468.16,0.99622,0.99444,0.99470,0.73197
12,Case 4: Dropout,2,dropout_0.1,32,32,0.01,0.0001,SGD,640,0.1,0,SiLU,462.76,0.99622,0.99444,0.99470,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train15, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

In [12]:
BEST_PARAMS["epochs"] = 32
BEST_PARAMS["weight_decay"] = 0.0001
BEST_PARAMS["batch"] = 32
case4_dropout()


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.994      0.995      0.732
Speed: 0.1ms preprocess, 1.7ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to runs/detect/train


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 4: Dropout,1,dropout_0,32,32,0.01,0.0001,SGD,640,0,0,SiLU,471.56,0.99622,0.99444,0.9947,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

...

,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 4: Dropout,1,dropout_0,32,32,0.01,0.0001,SGD,640,0,0,SiLU,471.56,0.99622,0.99444,0.9947,0.73197
1,Case 4: Dropout,2,dropout_0.1,32,32,0.01,0.0001,SGD,640,0.1,0,SiLU,464.40,0.99622,0.99444,0.9947,0.73197
2,Case 4: Dropout,3,dropout_0.2,32,32,0.01,0.0001,SGD,640,0.2,0,SiLU,463.41,0.99622,0.99444,0.9947,0.73197
3,Case 4: Dropout,4,dropout_0.3,32,32,0.01,0.0001,SGD,640,0.3,0,SiLU,457.36,0.99622,0.99444,0.9947,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.5, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.994      0.995      0.732
Speed: 0.2ms preprocess, 1.4ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to runs/detect/train5


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 4: Dropout,1,dropout_0,32,32,0.01,0.0001,SGD,640,0,0,SiLU,471.56,0.99622,0.99444,0.9947,0.73197
1,Case 4: Dropout,2,dropout_0.1,32,32,0.01,0.0001,SGD,640,0.1,0,SiLU,464.40,0.99622,0.99444,0.9947,0.73197
2,Case 4: Dropout,3,dropout_0.2,32,32,0.01,0.0001,SGD,640,0.2,0,SiLU,463.41,0.99622,0.99444,0.9947,0.73197
3,Case 4: Dropout,4,dropout_0.3,32,32,0.01,0.0001,SGD,640,0.3,0,SiLU,457.36,0.99622,0.99444,0.9947,0.73197
4,Case 4: Dropout,5,dropout_0.5,32,32,0.01,0.0001,SGD,640,0.5,0,SiLU,458.67,0.99622,0.99444,0.9947,0.73197


In [13]:
case6_optimizer()

Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.994      0.995      0.732
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to runs/detect/train6


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 4: Dropout,1,dropout_0,32,32,0.01,0.0001,SGD,640,0,0,SiLU,471.56,0.99622,0.99444,0.9947,0.73197
1,Case 4: Dropout,2,dropout_0.1,32,32,0.01,0.0001,SGD,640,0.1,0,SiLU,464.40,0.99622,0.99444,0.9947,0.73197
2,Case 4: Dropout,3,dropout_0.2,32,32,0.01,0.0001,SGD,640,0.2,0,SiLU,463.41,0.99622,0.99444,0.9947,0.73197
3,Case 4: Dropout,4,dropout_0.3,32,32,0.01,0.0001,SGD,640,0.3,0,SiLU,457.36,0.99622,0.99444,0.9947,0.73197
4,Case 4: Dropout,5,dropout_0.5,32,32,0.01,0.0001,SGD,640,0.5,0,SiLU,458.67,0.99622,0.99444,0.9947,0.73197
5,Case 6: Optimizer,1,optimizer_SGD,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,457.57,0.99622,0.99444,0.9947,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train7, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience

...

,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 4: Dropout,1,dropout_0,32,32,0.01,0.0001,SGD,640,0,0,SiLU,471.56,0.99622,0.99444,0.99470,0.73197
1,Case 4: Dropout,2,dropout_0.1,32,32,0.01,0.0001,SGD,640,0.1,0,SiLU,464.40,0.99622,0.99444,0.99470,0.73197
2,Case 4: Dropout,3,dropout_0.2,32,32,0.01,0.0001,SGD,640,0.2,0,SiLU,463.41,0.99622,0.99444,0.99470,0.73197
3,Case 4: Dropout,4,dropout_0.3,32,32,0.01,0.0001,SGD,640,0.3,0,SiLU,457.36,0.99622,0.99444,0.99470,0.73197
4,Case 4: Dropout,5,dropout_0.5,32,32,0.01,0.0001,SGD,640,0.5,0,SiLU,458.67,0.99622,0.99444,0.99470,0.73197
5,Case 6: Optimizer,1,optimizer_SGD,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,457.57,0.99622,0.99444,0.99470,0.73197
6,Case 6: Optimizer,2,optimizer_Adam,32,32,0.01,0.0001,Adam,640,0.0,0,SiLU,460.64,0.99253,0.99388,0.99452,0.71447
7,Case 6: Optimizer,3,optimizer_AdamW,32,32,0.01,0.0001,AdamW,640,0.0,0,SiLU,463.93,0.99375,0.99499,0.99433,0.72401
8,Case 6: Optimizer,4,optimizer_RMSProp,32,32,0.01,0.0001,RMSProp,640,0.0,0,SiLU,463.36,0.96645,0.94514,0.97844,0.54309


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train10, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adamax, overlap_mask=True, patie

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.995      0.994      0.994      0.732
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to runs/detect/train10


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 4: Dropout,1,dropout_0,32,32,0.01,0.0001,SGD,640,0,0,SiLU,471.56,0.99622,0.99444,0.99470,0.73197
1,Case 4: Dropout,2,dropout_0.1,32,32,0.01,0.0001,SGD,640,0.1,0,SiLU,464.40,0.99622,0.99444,0.99470,0.73197
2,Case 4: Dropout,3,dropout_0.2,32,32,0.01,0.0001,SGD,640,0.2,0,SiLU,463.41,0.99622,0.99444,0.99470,0.73197
3,Case 4: Dropout,4,dropout_0.3,32,32,0.01,0.0001,SGD,640,0.3,0,SiLU,457.36,0.99622,0.99444,0.99470,0.73197
4,Case 4: Dropout,5,dropout_0.5,32,32,0.01,0.0001,SGD,640,0.5,0,SiLU,458.67,0.99622,0.99444,0.99470,0.73197
5,Case 6: Optimizer,1,optimizer_SGD,32,32,0.01,0.0001,SGD,640,0.0,0,SiLU,457.57,0.99622,0.99444,0.99470,0.73197
6,Case 6: Optimizer,2,optimizer_Adam,32,32,0.01,0.0001,Adam,640,0.0,0,SiLU,460.64,0.99253,0.99388,0.99452,0.71447
7,Case 6: Optimizer,3,optimizer_AdamW,32,32,0.01,0.0001,AdamW,640,0.0,0,SiLU,463.93,0.99375,0.99499,0.99433,0.72401
8,Case 6: Optimizer,4,optimizer_RMSProp,32,32,0.01,0.0001,RMSProp,640,0.0,0,SiLU,463.36,0.96645,0.94514,0.97844,0.54309
9,Case 6: Optimizer,5,optimizer_Adamax,32,32,0.01,0.0001,Adamax,640,0.0,0,SiLU,458.61,0.99553,0.99388,0.99399,0.73196


In [16]:
BEST_PARAMS["epochs"] = 32
BEST_PARAMS["weight_decay"] = 0.0001
BEST_PARAMS["batch"] = 32
BEST_PARAMS["optimizer"] = "SGD"   # keep optimizer locked in from Case 6

case7_learning_rate()


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train11, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patienc

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.997      0.993      0.994      0.705
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to runs/detect/train11


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
1,Case 4: Dropout,2,dropout_0.1,32,32,0.010,0.0001,SGD,640,0.1,0,SiLU,464.40,0.99622,0.99444,0.99470,0.73197
2,Case 4: Dropout,3,dropout_0.2,32,32,0.010,0.0001,SGD,640,0.2,0,SiLU,463.41,0.99622,0.99444,0.99470,0.73197
3,Case 4: Dropout,4,dropout_0.3,32,32,0.010,0.0001,SGD,640,0.3,0,SiLU,457.36,0.99622,0.99444,0.99470,0.73197
4,Case 4: Dropout,5,dropout_0.5,32,32,0.010,0.0001,SGD,640,0.5,0,SiLU,458.67,0.99622,0.99444,0.99470,0.73197
5,Case 6: Optimizer,1,optimizer_SGD,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,457.57,0.99622,0.99444,0.99470,0.73197
6,Case 6: Optimizer,2,optimizer_Adam,32,32,0.010,0.0001,Adam,640,0.0,0,SiLU,460.64,0.99253,0.99388,0.99452,0.71447
7,Case 6: Optimizer,3,optimizer_AdamW,32,32,0.010,0.0001,AdamW,640,0.0,0,SiLU,463.93,0.99375,0.99499,0.99433,0.72401
8,Case 6: Optimizer,4,optimizer_RMSProp,32,32,0.010,0.0001,RMSProp,640,0.0,0,SiLU,463.36,0.96645,0.94514,0.97844,0.54309
9,Case 6: Optimizer,5,optimizer_Adamax,32,32,0.010,0.0001,Adamax,640,0.0,0,SiLU,458.61,0.99553,0.99388,0.99399,0.73196
10,Case 7: Learning Rate,1,lr_0.001,32,32,0.001,0.0001,SGD,640,0.0,0,SiLU,483.19,0.99727,0.99333,0.99369,0.70411


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train12, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patienc

...

,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
3,Case 4: Dropout,4,dropout_0.3,32,32,0.010,0.0001,SGD,640,0.3,0,SiLU,457.36,0.99622,0.99444,0.99470,0.73197
4,Case 4: Dropout,5,dropout_0.5,32,32,0.010,0.0001,SGD,640,0.5,0,SiLU,458.67,0.99622,0.99444,0.99470,0.73197
5,Case 6: Optimizer,1,optimizer_SGD,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,457.57,0.99622,0.99444,0.99470,0.73197
6,Case 6: Optimizer,2,optimizer_Adam,32,32,0.010,0.0001,Adam,640,0.0,0,SiLU,460.64,0.99253,0.99388,0.99452,0.71447
7,Case 6: Optimizer,3,optimizer_AdamW,32,32,0.010,0.0001,AdamW,640,0.0,0,SiLU,463.93,0.99375,0.99499,0.99433,0.72401
8,Case 6: Optimizer,4,optimizer_RMSProp,32,32,0.010,0.0001,RMSProp,640,0.0,0,SiLU,463.36,0.96645,0.94514,0.97844,0.54309
9,Case 6: Optimizer,5,optimizer_Adamax,32,32,0.010,0.0001,Adamax,640,0.0,0,SiLU,458.61,0.99553,0.99388,0.99399,0.73196
10,Case 7: Learning Rate,1,lr_0.001,32,32,0.001,0.0001,SGD,640,0.0,0,SiLU,483.19,0.99727,0.99333,0.99369,0.70411
11,Case 7: Learning Rate,2,lr_0.005,32,32,0.005,0.0001,SGD,640,0.0,0,SiLU,500.17,0.99666,0.99532,0.99466,0.72074
12,Case 7: Learning Rate,3,lr_0.01,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,494.95,0.99622,0.99444,0.99470,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train14, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.994      0.994      0.994      0.733
Speed: 0.1ms preprocess, 2.1ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to runs/detect/train14


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
4,Case 4: Dropout,5,dropout_0.5,32,32,0.010,0.0001,SGD,640,0.5,0,SiLU,458.67,0.99622,0.99444,0.99470,0.73197
5,Case 6: Optimizer,1,optimizer_SGD,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,457.57,0.99622,0.99444,0.99470,0.73197
6,Case 6: Optimizer,2,optimizer_Adam,32,32,0.010,0.0001,Adam,640,0.0,0,SiLU,460.64,0.99253,0.99388,0.99452,0.71447
7,Case 6: Optimizer,3,optimizer_AdamW,32,32,0.010,0.0001,AdamW,640,0.0,0,SiLU,463.93,0.99375,0.99499,0.99433,0.72401
8,Case 6: Optimizer,4,optimizer_RMSProp,32,32,0.010,0.0001,RMSProp,640,0.0,0,SiLU,463.36,0.96645,0.94514,0.97844,0.54309
9,Case 6: Optimizer,5,optimizer_Adamax,32,32,0.010,0.0001,Adamax,640,0.0,0,SiLU,458.61,0.99553,0.99388,0.99399,0.73196
10,Case 7: Learning Rate,1,lr_0.001,32,32,0.001,0.0001,SGD,640,0.0,0,SiLU,483.19,0.99727,0.99333,0.99369,0.70411
11,Case 7: Learning Rate,2,lr_0.005,32,32,0.005,0.0001,SGD,640,0.0,0,SiLU,500.17,0.99666,0.99532,0.99466,0.72074
12,Case 7: Learning Rate,3,lr_0.01,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,494.95,0.99622,0.99444,0.99470,0.73197
13,Case 7: Learning Rate,4,lr_0.05,32,32,0.050,0.0001,SGD,640,0.0,0,SiLU,493.74,0.99392,0.99444,0.99389,0.73307


In [17]:
BEST_PARAMS["epochs"] = 32
BEST_PARAMS["weight_decay"] = 0.0001
BEST_PARAMS["batch"] = 32
BEST_PARAMS["optimizer"] = "SGD"
BEST_PARAMS["lr0"] = 0.01   # keep best LR

case8_freeze_layers()


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train15, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.996      0.994      0.995      0.732
Speed: 0.1ms preprocess, 1.4ms inference, 0.0ms loss, 4.5ms postprocess per image
Results saved to runs/detect/train15


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
5,Case 6: Optimizer,1,optimizer_SGD,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,457.57,0.99622,0.99444,0.99470,0.73197
6,Case 6: Optimizer,2,optimizer_Adam,32,32,0.010,0.0001,Adam,640,0.0,0,SiLU,460.64,0.99253,0.99388,0.99452,0.71447
7,Case 6: Optimizer,3,optimizer_AdamW,32,32,0.010,0.0001,AdamW,640,0.0,0,SiLU,463.93,0.99375,0.99499,0.99433,0.72401
8,Case 6: Optimizer,4,optimizer_RMSProp,32,32,0.010,0.0001,RMSProp,640,0.0,0,SiLU,463.36,0.96645,0.94514,0.97844,0.54309
9,Case 6: Optimizer,5,optimizer_Adamax,32,32,0.010,0.0001,Adamax,640,0.0,0,SiLU,458.61,0.99553,0.99388,0.99399,0.73196
10,Case 7: Learning Rate,1,lr_0.001,32,32,0.001,0.0001,SGD,640,0.0,0,SiLU,483.19,0.99727,0.99333,0.99369,0.70411
11,Case 7: Learning Rate,2,lr_0.005,32,32,0.005,0.0001,SGD,640,0.0,0,SiLU,500.17,0.99666,0.99532,0.99466,0.72074
12,Case 7: Learning Rate,3,lr_0.01,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,494.95,0.99622,0.99444,0.99470,0.73197
13,Case 7: Learning Rate,4,lr_0.05,32,32,0.050,0.0001,SGD,640,0.0,0,SiLU,493.74,0.99392,0.99444,0.99389,0.73307
14,Case 8: Freeze Layers,1,freeze_0,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,513.66,0.99622,0.99444,0.99470,0.73197


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train16, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patienc

...

,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
7,Case 6: Optimizer,3,optimizer_AdamW,32,32,0.010,0.0001,AdamW,640,0.0,0,SiLU,463.93,0.99375,0.99499,0.99433,0.72401
8,Case 6: Optimizer,4,optimizer_RMSProp,32,32,0.010,0.0001,RMSProp,640,0.0,0,SiLU,463.36,0.96645,0.94514,0.97844,0.54309
9,Case 6: Optimizer,5,optimizer_Adamax,32,32,0.010,0.0001,Adamax,640,0.0,0,SiLU,458.61,0.99553,0.99388,0.99399,0.73196
10,Case 7: Learning Rate,1,lr_0.001,32,32,0.001,0.0001,SGD,640,0.0,0,SiLU,483.19,0.99727,0.99333,0.99369,0.70411
11,Case 7: Learning Rate,2,lr_0.005,32,32,0.005,0.0001,SGD,640,0.0,0,SiLU,500.17,0.99666,0.99532,0.99466,0.72074
12,Case 7: Learning Rate,3,lr_0.01,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,494.95,0.99622,0.99444,0.99470,0.73197
13,Case 7: Learning Rate,4,lr_0.05,32,32,0.050,0.0001,SGD,640,0.0,0,SiLU,493.74,0.99392,0.99444,0.99389,0.73307
14,Case 8: Freeze Layers,1,freeze_0,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,513.66,0.99622,0.99444,0.99470,0.73197
15,Case 8: Freeze Layers,2,freeze_10,32,32,0.010,0.0001,SGD,640,0.0,10,SiLU,452.43,0.99376,0.99333,0.99165,0.69899
16,Case 8: Freeze Layers,3,freeze_15,32,32,0.010,0.0001,SGD,640,0.0,15,SiLU,455.12,0.98482,0.99055,0.99111,0.68906


Ultralytics 8.3.189 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-text-detection-data2/YOLO_text_detection_data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=22, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train18, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patienc

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        300       1798      0.952      0.961      0.984      0.607
Speed: 0.1ms preprocess, 1.9ms inference, 0.0ms loss, 3.9ms postprocess per image
Results saved to runs/detect/train18


,Case,Exp No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
8,Case 6: Optimizer,4,optimizer_RMSProp,32,32,0.010,0.0001,RMSProp,640,0.0,0,SiLU,463.36,0.96645,0.94514,0.97844,0.54309
9,Case 6: Optimizer,5,optimizer_Adamax,32,32,0.010,0.0001,Adamax,640,0.0,0,SiLU,458.61,0.99553,0.99388,0.99399,0.73196
10,Case 7: Learning Rate,1,lr_0.001,32,32,0.001,0.0001,SGD,640,0.0,0,SiLU,483.19,0.99727,0.99333,0.99369,0.70411
11,Case 7: Learning Rate,2,lr_0.005,32,32,0.005,0.0001,SGD,640,0.0,0,SiLU,500.17,0.99666,0.99532,0.99466,0.72074
12,Case 7: Learning Rate,3,lr_0.01,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,494.95,0.99622,0.99444,0.99470,0.73197
13,Case 7: Learning Rate,4,lr_0.05,32,32,0.050,0.0001,SGD,640,0.0,0,SiLU,493.74,0.99392,0.99444,0.99389,0.73307
14,Case 8: Freeze Layers,1,freeze_0,32,32,0.010,0.0001,SGD,640,0.0,0,SiLU,513.66,0.99622,0.99444,0.99470,0.73197
15,Case 8: Freeze Layers,2,freeze_10,32,32,0.010,0.0001,SGD,640,0.0,10,SiLU,452.43,0.99376,0.99333,0.99165,0.69899
16,Case 8: Freeze Layers,3,freeze_15,32,32,0.010,0.0001,SGD,640,0.0,15,SiLU,455.12,0.98482,0.99055,0.99111,0.68906
17,Case 8: Freeze Layers,4,freeze_22,32,32,0.010,0.0001,SGD,640,0.0,22,SiLU,439.20,0.95175,0.96107,0.98368,0.60693


ValueError: Invalid export format='yaml'. Valid formats are ('torchscript', 'onnx', 'openvino', 'engine', 'coreml', 'saved_model', 'pb', 'tflite', 'edgetpu', 'tfjs', 'paddle', 'mnn', 'ncnn', 'imx', 'rknn')

In [18]:
import re
from pathlib import Path

# Path to the exported yaml
base_yaml = Path("yolov8n.yaml").read_text()

# List of activations to test
activations = ["SiLU", "ReLU", "LeakyReLU", "Mish", "GELU"]

# Prepare custom yaml files
for act in activations:
    custom_yaml = re.sub(r"act:\s*\w+", f"act: {act}", base_yaml)
    out_file = f"yolov8n_{act}.yaml"
    Path(out_file).write_text(custom_yaml)
    print(f"Saved {out_file}")


FileNotFoundError: [Errno 2] No such file or directory: 'yolov8n.yaml'